# TP 3 — Nettoyer et tester : un module de transformations PySpark
**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Objectif : transformer `customers.csv` (sale) en une table clients **propre**,
avec un code **modulaire et testé**.

**Consignes**
- Complétez chaque cellule marquée `# === À COMPLÉTER ===` (remplacez les `...`).
- Exécutez le notebook **de bout en bout** sans erreur.
- Poussez le notebook **avec ses sorties** sur votre dépôt GitHub.

Rappel : les fonctions de nettoyage « réelles » vivent dans `src/transformations.py`.
Ce notebook **démontre** et **mesure** ; il importe le module.


## 0. Vérification de l'environnement


In [1]:
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F

print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("TP3-nettoyage")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark


Python  : 3.12.10
PySpark : 4.2.0


C:\Users\amedt\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


### Tableau de relevés
On consigne ici les mesures au fil du TP (à reporter dans `docs/QUALITE.md`).


In [2]:
releves = {
    "lignes_brutes": None,
    "emails_manquants": None,
    "villes_distinctes_avant": None,
    "villes_distinctes_apres": None,
    "doublons_exacts": None,
    "lignes_apres_nettoyage": None,
}
releves


{'lignes_brutes': None,
 'emails_manquants': None,
 'villes_distinctes_avant': None,
 'villes_distinctes_apres': None,
 'doublons_exacts': None,
 'lignes_apres_nettoyage': None}

## 1. Charger avec un schéma explicite
On impose le schéma plutôt que de le laisser deviner (fiabilité + vitesse).


In [3]:
import sys
import pathlib

# Rend "src" importable que le notebook soit lance avec pour repertoire de
# travail la racine du projet ou le dossier notebooks/.
RACINE = pathlib.Path.cwd() if (pathlib.Path.cwd() / "src").exists() else pathlib.Path.cwd().parent
sys.path.append(str(RACINE))

from src.transformations import lire_clients

df_brut = lire_clients(spark, str(RACINE / "data" / "customers.csv"))

releves["lignes_brutes"] = df_brut.count()
df_brut.printSchema()
print("lignes :", releves["lignes_brutes"])

C:\Users\amedt\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\amedt\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\pyspark\sql\udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


root
 |-- customer_id: string (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telephone: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- region: string (nullable = true)
 |-- date_naissance: string (nullable = true)
 |-- date_inscription: string (nullable = true)

lignes : 5025


## 2. Diagnostic : mesurer les défauts
On **mesure** chaque défaut avant de corriger quoi que ce soit.

### 2.1 Valeurs manquantes par colonne


In [4]:
# nulls par colonne
df_brut.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_brut.columns
]).show()

+-----------+------+---+-----+---------+-------+-----+------+--------------+----------------+
|customer_id|prenom|nom|email|telephone|adresse|ville|region|date_naissance|date_inscription|
+-----------+------+---+-----+---------+-------+-----+------+--------------+----------------+
|          0|     0|  0|   75|        0|      0|    0|     0|             0|               0|
+-----------+------+---+-----+---------+-------+-----+------+--------------+----------------+



### 2.2 Faux manquants (emails "" ou "N/A")


In [5]:
nb_email_vide = df_brut.filter(
    F.col("email").isNull() | F.trim(F.col("email")).isin("", "N/A")
).count()
print("emails vides ou N/A :", nb_email_vide)
releves["emails_manquants"] = nb_email_vide

emails vides ou N/A : 150


### 2.3 Villes distinctes (avant normalisation) et doublons exacts


In [6]:
releves["villes_distinctes_avant"] = df_brut.select("ville").distinct().count()
releves["doublons_exacts"] = df_brut.count() - df_brut.distinct().count()
print(releves["villes_distinctes_avant"], "villes distinctes (brut)")
print(releves["doublons_exacts"], "doublons exacts")

56 villes distinctes (brut)
15 doublons exacts


## 3. Les fonctions de transformation (dans src/)
Les fonctions vivent dans `src/transformations.py`, où elles sont **testées**
(`tests/test_transformations.py`, `pytest -q`). Le notebook se contente de les
**importer** et de les démontrer sur les données réelles — pas de définition
en double qui risquerait de diverger du code testé.

In [7]:
from src.transformations import unifier_manquants, normaliser_email

### 3.2 Ville (avec retrait d'accents)

In [8]:
from src.transformations import normaliser_ville

### 3.3 Téléphone et date de naissance


In [9]:
from src.transformations import normaliser_telephone, valider_naissance

### 3.4 Déduplication (après normalisation)


In [10]:
from src.transformations import dedupliquer_clients

## 4. Assembler le pipeline et mesurer l'effet


In [11]:
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
from pyspark.sql import SparkSession, functions as F

print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("TP3-nettoyage")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark

Python  : 3.12.10
PySpark : 4.2.0


In [12]:
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
from pyspark.sql import SparkSession, functions as F

java_opts = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
)

print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("TP3-nettoyage")
         .config("spark.driver.extraJavaOptions", java_opts)
         .config("spark.executor.extraJavaOptions", java_opts)
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark

Python  : 3.12.10
PySpark : 4.2.0


In [30]:
    from src.transformations import nettoyer_clients

    df_net = nettoyer_clients(df_brut)

        releves["villes_distinctes_apres"] = df_net.select("ville_norm").distinct().count()
        releves["lignes_apres_nettoyage"]  = df_net.count()
        print("avant :", releves["lignes_brutes"], "-> apres :", releves["lignes_apres_nettoyage"])
        print("villes distinctes :", releves["villes_distinctes_avant"], "-> apres :", releves["villes_distinctes_apres"])

IndentationError: unexpected indent (1146571000.py, line 5)

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

### 4.1 Vérification visuelle : top des villes après nettoyage


In [16]:
df_net.groupBy("ville_norm").count().orderBy(F.desc("count")).show(10)

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

### 4.2 Tableau de relevés final


In [14]:
for k, v in releves.items():
    print(f"{k:30s} : {v}")


lignes_brutes                  : 5025
emails_manquants               : 150
villes_distinctes_avant        : 56
villes_distinctes_apres        : None
doublons_exacts                : 15
lignes_apres_nettoyage         : None


## 5. Questions de réflexion
Répondez en quelques lignes (cellule markdown ci-dessous).

1. Combien de villes distinctes **avant** et **après** normalisation ? Que
   conclure sur l'effet de la casse et des accents ?
2. Quelle décision avez-vous prise pour les emails manquants (drop ou fill) ?
   Pourquoi ?
3. Vous avez écrit une **UDF** (`sans_accent`). À quel coût ? Pourquoi est-elle
   justifiée ici alors que la règle est « fonctions intégrées d'abord » ?
4. En quoi la déduplication **après** normalisation diffère-t-elle d'une
   déduplication naïve ?


* Réponse :

1. Avant normalisation, `ville` contient des variantes de casse et d'accents
   (« DAKAR », « thies », « Thiès »…), ce qui gonfle artificiellement le
   nombre de villes distinctes. Après normalisation (`ville_norm` : trim +
   initcap + suppression des accents), toutes les variantes d'une même ville
   se regroupent sous une seule clé — le nombre de villes distinctes chute
   nettement (voir `releves` ci-dessus pour les valeurs mesurées sur ces
   données).

2. Choix : les emails manquants (`""`, `"N/A"`) sont mis à `null` plutôt que
   supprimés (`unifier_manquants`), et un drapeau `email_valide` est ajouté
   plutôt que de rejeter la ligne. Raison : un email manquant ou mal formé
   n'invalide pas le reste des informations du client (téléphone, ville,
   date de naissance) ; supprimer la ligne ferait perdre ces autres données
   exploitables. Une autre équipe pourrait choisir de `drop` si l'email est
   une clé métier obligatoire pour son cas d'usage (envoi de campagnes email
   par exemple).

3. La fonction `sans_accent` est une UDF Python : chaque valeur repasse par
   l'interpréteur Python (sérialisation JVM <-> Python ligne par ligne), ce
   qui est nettement plus lent qu'une fonction Spark native vectorisée. Elle
   est justifiée ici car aucune fonction native `F.*` ne retire les accents
   Unicode (contrairement à `trim`/`initcap`/`lower` qui existent nativement
   et sont utilisées en priorité) ; le volume de données (échelle 0.1) rend
   ce coût négligeable en pratique.

4. Dédupliquer après normalisation regroupe les quasi-doublons qui ne
   diffèrent que par la casse de l'email (`Jean.DUPONT@Gmail.com` vs
   `jean.dupont@gmail.com`) : une fois l'email mis en minuscules, les deux
   lignes deviennent identiques sur la clé et `dedupliquer_clients` les
   fusionne. Une déduplication naïve (avant normalisation, ou sur
   `customer_id` seul) laisserait survivre ces quasi-doublons alors qu'ils
   représentent la même personne.

## 6. Vers le livrable
1. Déplacez les fonctions de la section 3 dans `src/transformations.py`.
2. Écrivez les tests dans `tests/test_transformations.py` (+ `conftest.py`).
3. Vérifiez `pytest -q` : **tout au vert**.
4. Remplissez `docs/QUALITE.md` avec le tableau de relevés.
5. Poussez le tout :

```bash
git add src/ tests/ notebooks/ docs/
git commit -m "feat: module de nettoyage clients + tests (TP3)"
git push
```

> Rappel : `data/` n'est **jamais** commité.


In [15]:
# Arret propre de la session Spark
spark.stop()
print("Session fermee. Notebook termine.")


Session fermee. Notebook termine.
